In [12]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect(':memory:')

# ----------------------------------------------------
# 1. df_coupons：领券明细表（故意乱序、带幽灵空格）
# ----------------------------------------------------
coupon_data = {
    'coupon_id':   [10,            20,            30],
    'user_id':     [901,           901,           902],
    # 👈 脏数据：前后空格、乱序
    'claim_time':  ['2026-06-06 12:00:00', ' 2026-06-06 18:00:00 ', '2026-06-06 14:00:00']
}
df_coupons = pd.DataFrame(coupon_data)
df_coupons.to_sql('coupons', conn, index=False, if_exists='replace')

# ----------------------------------------------------
# 2. df_orders：交易明细表（一个用户有多笔交易，时间错综复杂）
# ----------------------------------------------------
order_data = {
    'order_id':    [5001,  5002,  5003,  5004,  5005],
    'user_id':     [901,   901,   901,   902,   902],
    # 👈 901号在 12:05 下单（属于10号券的10分钟窗口）
    # 👈 901号在 12:15 下单（超出了10分钟窗口，合法）
    # 👈 901号在 18:02 下单（属于20号券的10分钟窗口）
    'order_time':  ['2026-06-06 12:05:00', '2026-06-06 12:15:00', '2026-06-06 18:02:00', '2026-06-06 14:05:00', '2026-06-06 14:30:00']
}
df_orders = pd.DataFrame(order_data)
df_orders.to_sql('orders', conn, index=False, if_exists='replace')

print("====== ⚔️ 第五宇宙多路风控大盘已锁死 =======")
print("原始领券表 coupons：")
print(df_coupons)
print("\n原始订单表 orders：")
print(df_orders)

====== ⚔️ 第五宇宙多路风控大盘已锁死 =======
原始领券表 coupons：
   coupon_id  user_id             claim_time
0         10      901    2026-06-06 12:00:00
1         20      901   2026-06-06 18:00:00 
2         30      902    2026-06-06 14:00:00

原始订单表 orders：
   order_id  user_id           order_time
0      5001      901  2026-06-06 12:05:00
1      5002      901  2026-06-06 12:15:00
2      5003      901  2026-06-06 18:02:00
3      5004      902  2026-06-06 14:05:00
4      5005      902  2026-06-06 14:30:00


📋 业务需求：
风控总监要求，同时在 SQL 轨道 和 Pandas 轨道 完成以下合围：

揪出所有“在领券后 10 分钟内（含 10 分钟）就火速下单”的欺诈嫌疑订单。算出每个高危用户符合这种欺诈特征的【总订单数】以及【最早作案的领券时间（Min Claim Time）】。最终输出不带索引。

📊 输出字段要求：
最终两轨道的报表必须 100% 镜像对齐：
user_id | fraud_order_count | earliest_fraud_claim_time

In [ ]:
# ====================================================
# 🧱 SQL 轨道 - 拨乱反正·无懈可击完全体
# ====================================================
sql_query = """
WITH coupon_cleaned AS (
    -- 👑 先用 TRIM 物理剥离空格凡胎，再用 datetime 熔炼重塑金身！
    SELECT coupon_id, user_id, datetime(TRIM(claim_time)) AS claim_time
    FROM coupons
),
order_cleaned AS (
    -- 订单表同步执行强类型格式化
    SELECT order_id, user_id, datetime(TRIM(order_time)) AS order_time
    FROM orders
),
fraud_matches AS (
    -- ⚔️ 时空大爆炸天网
    SELECT o.user_id, o.order_id, c.claim_time, o.order_time
    FROM order_cleaned AS o
    INNER JOIN coupon_cleaned AS c ON o.user_id = c.user_id 
        AND o.order_time >= c.claim_time
        AND o.order_time <= datetime(c.claim_time, '+10 minutes')
)
SELECT 
    user_id,
    COUNT(order_id) AS fraud_order_count,
    MIN(claim_time) AS earliest_fraud_claim_time
FROM fraud_matches
GROUP BY user_id
ORDER BY fraud_order_count DESC;
"""

df_sql = pd.read_sql_query(sql_query, conn)
print("=== 🛡️ 听取克星意见修正后：真理大盘落地 ===")
print(df_sql.to_string(index=False))

=== 🛡️ 听取克星意见修正后：真理大盘落地 ===
 user_id  fraud_order_count earliest_fraud_claim_time
     901                  2       2026-06-06 12:00:00
     902                  1       2026-06-06 14:00:00


In [9]:
# ====================================================
# 👑 PANDAS 轨道 - 最终无自爆、无漏单完全体风控流水线
# ====================================================
import pandas as pd
# 1. ⚔️ 强力前置洗涤与时空整队：领券表（右表）
# 💡 铁律：merge_asof 强制要求右表时间轴必须是干净的【升序排列】！
df_coupons_sorted = (
    df_coupons
    .assign(
        claim_time = lambda df: pd.to_datetime(df['claim_time'].str.strip()) # 剥离洋葱：先洗净，后升级
    )
    .sort_values(by='claim_time', ascending=True)
)

# 2. ⚔️ 强力前置洗涤与时空整队：订单表（左表 / 主时间表）
# 💡 铁律：左表时间轴也必须死死焊在【升序排列】轨道上！
df_orders_sorted = (
    df_orders
    .assign(
        order_time = lambda df: pd.to_datetime(df['order_time'])
    )
    .sort_values(by='order_time', ascending=True)
)

# 3. 📡 祭出终极时空游标：流式近似合围（行数绝对不膨胀，彻底免疫 OOM）
# 💡 翻阅 Obsidian 圣经：左表为主，雷达向过去看（backward），在同一人（by='user_id'）舱内精准吸附
df_asof_merge = pd.merge_asof(
    df_orders_sorted,             # 👑 左表：主时间传送带。决定最终输出的物理行数骨架！
    df_coupons_sorted,            # 🧱 右表：被动匹配表（时空能量站）。
    left_on='order_time',         # 左表的时间戳对齐轴
    right_on='claim_time',        # 右表的时间戳对齐轴
    by='user_id',                 # 隔离舱：必须在同一个人头里玩时空平移
    direction='backward'          # 雷达向过去看，寻找订单发生前、最邻近的那张领券时间
)

# 4. 🗜️ 挂上 10 分钟天网，执行终极降维收割
# 🧱 极客流：纯净的、无函数嵌套的时空截断
df_final = (
    df_asof_merge
    .assign(
        time_diff = lambda df: df['order_time'] - df['claim_time']
    )
    # 🔥 核心修正：不在 query 里调用 pd，直接跟纯粹的时间跨度（Timestamp/Timedelta字符串）或者变量对齐
    # 其实 Pandas 在 query 里支持直接解析标准的字符串时间跨度：
    .query("time_diff >= '0 days' and time_diff <= '10 minutes'") 
    
    .groupby('user_id')
    .agg(
        fraud_order_count=('order_id', 'count'),
        earliest_fraud_claim_time=('claim_time', 'min')
    )
    .reset_index()
    .sort_values(by='fraud_order_count', ascending=False)
)

print("=== 🛡️ 最终交付：PANDAS 轨道完全体流式风控报表 ===")
print(df_final.to_string(index=False))

=== 🛡️ 最终交付：PANDAS 轨道完全体流式风控报表 ===
 user_id  fraud_order_count earliest_fraud_claim_time
     901                  2       2026-06-06 12:00:00
     902                  1       2026-06-06 14:00:00
